In [0]:
%sql

CREATE TABLE IF NOT EXISTS mba.trusted.f_geracao_energia_mensal
(
    id_usina BIGINT COMMENT 'Chave substituta da usina, proveniente da dimensão d_usina',
    ANO_MES INT COMMENT 'Ano e mês de referência da geração no formato YYYYMM',
    GERACAO DECIMAL(20,6) COMMENT 'Geração de energia elétrica acumulada no mês, conforme dados horários do ONS'
)
USING DELTA
COMMENT 'Fato de geração mensal de energia elétrica por usina, consolidada a partir dos dados horários do ONS';

In [0]:
%sql
MERGE INTO mba.trusted.f_geracao_energia_mensal AS tgt
USING(
    SELECT b.id_usina,
        CAST(date_format(a.din_instante, 'yyyyMM')AS INT) AS ANO_MES,
        CAST(COALESCE(SUM(a.val_geracao),0) AS DECIMAL(20,6)) AS GERACAO
    FROM mba.raw.ons_geracao_usina_horaria a
    INNER JOIN mba.trusted.d_usinas b ON substring_index(TRIM(a.ceg),'-',1) = b.CodCEG
    GROUP BY b.id_usina, date_format(a.din_instante, 'yyyyMM')
) AS src
ON  tgt.id_usina = src.id_usina AND tgt.ANO_MES  = src.ANO_MES

WHEN NOT MATCHED THEN
    INSERT(id_usina, ANO_MES, GERACAO)
    VALUES(src.id_usina, src.ANO_MES, src.GERACAO);

In [0]:
dbutils.notebook.exit("stop")

In [0]:
%sql
select *
from mba.trusted.f_geracao_energia_mensal a
join mba.trusted.d_usinas b on a.id_usina=b.id_usina
where NomEmpreendimento like 'Belo%'